In [ ]:
import numpy as np
import pandas as pd


In [ ]:
df = pd.read_csv('/Users/kaylamullen/Desktop/PITNE/Data/merged/merged_full.csv')

In [ ]:
df_demo = pd.read_csv('/Users/kaylamullen/Desktop/PITNE/Data/merged/merged_demo.csv')

# ARCGIS Prep

In [ ]:
df.columns

In [ ]:
non_white_mapping = {
    'Hispanic or Latino (of all races)' : 1, 
    'Asian' : 1, 
    'Black or African American' : 1, 
    'Middle Eastern or North African' : 1,
    'Native American or Alaskan Native' : 1,
    'Choose not to answer/Missing/Other' : -1,
    'White' : 0,
    'Multiple Races/Ethnicities' : 1,
}

df['is_non-white_household'] = df['race_simplified'].map(non_white_mapping)

In [ ]:
# Assuming your town column is named 'property_town'
applicant_counts = df['property_town'].value_counts().reset_index()
applicant_counts.columns = ['town', 'num_applications']

# Save it to CSV for upload to ArcGIS
applicant_counts.to_csv('/Users/kaylamullen/Desktop/PITNE/Data/geo_data/21_25_applications_by_town.csv', index=False)


In [ ]:
valid_race_apps = df[
    (df['race_simplified'].notna()) &
    (df['race_simplified'] != 'Choose not to answer/Missing/Other') &
    (df['latitude'].notna()) &
    (df['longitude'].notna())
].copy()

In [ ]:
valid_race_apps['race_simplified'].value_counts()

In [ ]:
df['race_simplified'].value_counts()

In [ ]:
valid_race_apps.to_csv('/Users/kaylamullen/Desktop/PITNE/Data/arcGIS_prepped/valid_race_applications_21_25.csv', index=False)


# Bivarate Chloropleth Map for White vs Non-White

In [ ]:
# Filter valid race entries
valid_df = df[df['is_non-white_household'] >= 0].copy()

In [ ]:
# Count number of apps per town and race
non_white_by_town = valid_df.groupby(['property_town', 'is_non-white_household']).size().unstack(fill_value=0)
non_white_by_town.columns = ['White Apps', 'Non-White Apps']
non_white_by_town

In [ ]:
# Percentages
non_white_by_town['total_apps'] = non_white_by_town['White Apps'] + non_white_by_town['Non-White Apps']
non_white_by_town['pct_white'] = (non_white_by_town['White Apps'] / non_white_by_town['total_apps'] * 100).round(1)
non_white_by_town['pct_Non-White'] = (non_white_by_town['Non-White Apps'] / non_white_by_town['total_apps'] * 100).round(1)
non_white_by_town

In [ ]:

# Reset index to get town column
non_white_by_town = non_white_by_town.reset_index()
non_white_by_town.to_csv('/Users/kaylamullen/Desktop/PITNE/Data/arcGIS_prepped/town_bivariate_map.csv', index=False)

# Bivariate Chloropleth % of all apps from white households vs % of all apps from HOC

In [ ]:
# Count number of white and non-white applications by town
wnw_by_percent = valid_df.groupby(['property_town', 'is_non-white_household']).size().unstack(fill_value=0)
wnw_by_percent.columns = ['White Apps', 'Non-White Apps']


In [ ]:
total_white = wnw_by_percent['White Apps'].sum()
total_nonwhite = wnw_by_percent['Non-White Apps'].sum()

In [ ]:
wnw_by_percent['Percent of all White Applications'] = (wnw_by_percent['White Apps'] / total_white * 100).round(2)
wnw_by_percent['Percent of all Non-White Applications'] = (wnw_by_percent['Non-White Apps'] / total_nonwhite * 100).round(2)


In [ ]:
wnw_by_percent = wnw_by_percent.reset_index()
wnw_by_percent


In [ ]:
def clean_column_name(col):
    col = col.replace('_', ' ')                      # underscores → spaces
    col = col.replace('pct', 'Percent')              # pct → Percent
    col = col.replace('apps', 'Applications')        # apps → Applications
    col = col.replace('nonwhite', 'Non-White')       # nonwhite → Non-White
    col = col.replace('white', 'White')              # white → White
    col = col.replace('property town', 'Town')       # optional simplification
    col = col.strip().title()                        # title-case everything
    return col


In [ ]:
df.columns = [clean_column_name(col) for col in df.columns]


In [ ]:
wnw_by_percent.columns = [clean_column_name(col) for col in wnw_by_percent.columns]


In [ ]:
wnw_by_percent.to_csv("/Users/kaylamullen/Desktop/PITNE/Data/arcGIS_prepped/by_town_wnw_percent_formatted.csv", index=False)


# current residence

In [ ]:
df_demo['curr_res_full'] = df_demo['curr_res_town'] + ", " + df_demo['curr_res_state']


In [ ]:
df_demo.to_csv("/Users/kaylamullen/Desktop/PITNE/Data/arcGIS_prepped/merged_demo_curr_res.csv", index=False)
